In [1]:
import sys
import time
import tracemalloc
from pathlib import Path
raiz = Path.cwd()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
from src.cuantizacion import (ajustar_cuantizador, transformar, reconstruir, calcular_error,
                              cuantizar_dataframe, reconstruir_dataframe,
                              guardar_modelo, cargar_modelo, guardar_modelos, cargar_modelos,
                              ESTRATEGIAS, REPRESENTANTES)

DIR_TABLAS = raiz / "resultados" / "actividad3" / "tablas"
DIR_MODELOS = raiz / "resultados" / "actividad3" / "modelos"
DIR_TABLAS.mkdir(parents=True, exist_ok=True)
DIR_MODELOS.mkdir(parents=True, exist_ok=True)
COLS = ["edad", "ingresos", "altura"]

In [2]:
x = np.random.default_rng(0).lognormal(size=100_000)
m = ajustar_cuantizador(x, "cuantiles", 10, columna="demo")
ruta = DIR_MODELOS / "demo_cuantiles_10.json"
guardar_modelo(m, ruta)
m2 = cargar_modelo(ruta)

print("límites idénticos:", np.array_equal(m.limites, m2.limites))
print("representantes idénticos:", all(np.array_equal(m.representantes[r], m2.representantes[r])
                                       for r in REPRESENTANTES))
print("códigos idénticos:", np.array_equal(transformar(x, m), transformar(x, m2)))
print(ruta.read_text(encoding="utf-8")[:450])

límites idénticos: True
representantes idénticos: True
códigos idénticos: True
{
  "version": 1,
  "columna": "demo",
  "estrategia": "cuantiles",
  "n_bins_solicitado": 10,
  "n_bins_real": 10,
  "limites": [
    0.011174542932887273,
    0.2781395653578184,
    0.4308692110284752,
    0.5923667705140268,
    0.7746559870718582,
    0.9958615941644124,
    1.2920394657935161,
    1.69525717578075,
    2.3133598420176096,
    3.586124003349102,
    113.5175770304286
  ],
  "representantes": {
    "punto_medio": [
      0.14


In [3]:
df = pd.read_parquet(raiz / "data" / "dataset_1M.parquet", columns=COLS)
df = df.sample(frac=1.0, random_state=0).reset_index(drop=True)
mitad = len(df) // 2
ajuste, prueba = df.iloc[:mitad], df.iloc[mitad:]

filas = []
for estrategia in ESTRATEGIAS:
    for k in (10, 50):
        _, modelos = cuantizar_dataframe(ajuste, COLS, estrategia, k)
        ruta = DIR_MODELOS / f"reutilizacion_1M_{estrategia}_{k}.json"
        guardar_modelos(modelos, ruta)
        cargados = cargar_modelos(ruta)                 # se aplica el modelo leído del JSON

        for c in COLS:
            m = cargados[c]
            antes = m.limites.copy()
            cod_aj = transformar(ajuste[c], m)
            cod_pr, diag = transformar(prueba[c], m, devolver_diagnostico=True)
            assert np.array_equal(antes, m.limites)                          # el modelo no cambió
            assert np.array_equal(cod_pr, transformar(prueba[c], modelos[c]))  # igual al modelo en memoria
            for rep in REPRESENTANTES:
                e_aj = calcular_error(ajuste[c], reconstruir(cod_aj, m, rep))
                e_pr = calcular_error(prueba[c], reconstruir(cod_pr, m, rep))
                filas.append({
                    "columna": c, "estrategia": estrategia, "bins": k,
                    "bins_reales": m.n_bins_real, "representante": rep,
                    "mae_ajuste": e_aj["mae"], "mae_prueba": e_pr["mae"],
                    "ratio_mae": e_pr["mae"] / e_aj["mae"],
                    "rmse_ajuste": e_aj["rmse"], "rmse_prueba": e_pr["rmse"],
                    "pct_fuera_rango_prueba": float(diag["fuera_rango"].mean() * 100),
                })

reut = pd.DataFrame(filas)
reut.to_csv(DIR_TABLAS / "reutilizacion_1M.csv", index=False)
reut.groupby(["columna", "estrategia", "bins"])[["mae_ajuste", "mae_prueba", "ratio_mae",
                                                  "pct_fuera_rango_prueba"]].mean().round(5)

mae_ajuste  mae_prueba  ratio_mae  \
columna  estrategia bins                                      
altura   cuantiles  10       1.98586     1.98214    0.99682   
                    50       0.41987     0.41619    0.99232   
         uniforme   10       2.41359     2.41329    0.99987   
                    50       0.49889     0.49953    1.00128   
edad     cuantiles  10       2.16754     2.17136    1.00164   
                    50       0.46172     0.46128    0.99912   
         uniforme   10       2.59489     2.59368    0.99954   
                    50       0.53429     0.53332    0.99818   
ingresos cuantiles  10    4081.07551  4061.57318    0.99732   
                    50     859.44556   856.95061    0.99885   
         uniforme   10    4538.92975  4544.36150    1.00126   
                    50    1071.74533  1072.89262    1.00107   

                          pct_fuera_rango_prueba  
columna  estrategia bins                          
altura   cuantiles  10                    0.0000  
                    50                    0.0000  
         uniforme   10                    0.0000  
                    50                    0.0000  
edad     cuantiles  10                    0.0014  
                    50                    0.0014  
         uniforme   10                    0.0014  
                    50                    0.0014  
ingresos cuantiles  10                    0.0002  
                    50                    0.0002  
         uniforme   10                    0.0002  
                    50                    0.0002

In [4]:
def medir(fn):
    """Devuelve (resultado, segundos, pico de memoria en MB). Ejecuta fn dos veces:
    una para medir tiempo y otra bajo tracemalloc para medir memoria."""
    t0 = time.perf_counter()
    res = fn()
    t = time.perf_counter() - t0
    tracemalloc.start()
    fn()
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return res, t, pico / 1e6

t0 = time.perf_counter()
df5 = pd.read_parquet(raiz / "data" / "dataset_5M.parquet", columns=COLS)
t_lectura = time.perf_counter() - t0
piloto = df5.sample(frac=0.10, random_state=0).reset_index(drop=True)
del df5
print(f"lectura 5M: {t_lectura:.2f} s | piloto: {len(piloto):,} filas, "
      f"{piloto.memory_usage(deep=True).sum() / 1e6:.0f} MB")

filas = []
for estrategia in ESTRATEGIAS:
    for k in (5, 10, 25, 50):
        (df_cod, modelos), t_cu, mem_cu = medir(lambda: cuantizar_dataframe(piloto, COLS, estrategia, k))
        t_rec_total, mem_rec = 0.0, 0.0
        for rep in REPRESENTANTES:
            _, t_r, m_r = medir(lambda: reconstruir_dataframe(df_cod, modelos, rep))
            t_rec_total += t_r
            mem_rec = max(mem_rec, m_r)
        filas.append({
            "estrategia": estrategia, "bins": k, "filas_piloto": len(piloto),
            "t_cuantizar_s": t_cu, "pico_MB_cuantizar": mem_cu,
            "t_reconstruir_3rep_s": t_rec_total, "pico_MB_reconstruir": mem_rec,
            "t_estimado_5M_s": (t_cu + t_rec_total) * 10,
            "pico_estimado_5M_MB": max(mem_cu, mem_rec) * 10,
        })

pil = pd.DataFrame(filas)
pil.to_csv(DIR_TABLAS / "piloto_5M.csv", index=False)
try:
    import os, psutil
    print("memoria total del proceso (MB):", round(psutil.Process(os.getpid()).memory_info().rss / 1e6))
except ImportError:
    pass
pil.round(3)

lectura 5M: 0.17 s | piloto: 500,000 filas, 12 MB
memoria total del proceso (MB): 338


,estrategia,bins,filas_piloto,t_cuantizar_s,pico_MB_cuantizar,t_reconstruir_3rep_s,pico_MB_reconstruir,t_estimado_5M_s,pico_estimado_5M_MB
0,uniforme,5,500000,0.080,32.705,0.034,19.008,1.133,327.046
1,uniforme,10,500000,0.096,31.369,0.034,19.008,1.303,313.693
2,uniforme,25,500000,0.116,30.586,0.033,19.008,1.489,305.864
3,uniforme,50,500000,0.130,30.310,0.032,19.008,1.626,303.099
4,cuantiles,5,500000,0.112,30.815,0.037,19.008,1.497,308.150
5,cuantiles,10,500000,0.138,30.416,0.040,19.008,1.780,304.158
6,cuantiles,25,500000,0.155,30.178,0.035,19.008,1.899,301.783
7,cuantiles,50,500000,0.181,30.101,0.033,19.008,2.140,301.011
